## Calling API from OpneRouter


In [1]:
!pip install openai requests -q

In [2]:
import os
import json
import re
import time
import requests
import subprocess
from openai import OpenAI
from google.colab import userdata

os.environ["OPENROUTER_API_KEY"] = userdata.get("OPENROUTER_API_KEY")

client = OpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key=os.environ["OPENROUTER_API_KEY"],
)
FREE_MODEL = "openrouter/free"
TOOL_MODEL = "openrouter/free"

### Your first API call

This is **inference**. You send tokens, get tokens back.  the API just wraps it with HTTP.

In [3]:
response = client.chat.completions.create(
    model=FREE_MODEL,
    messages=[{"role": "user", "content": "Hii"}],
    temperature=0.0,
    max_tokens=50,
)

print("Response:", response.choices[0].message.content)
print(f"Tokens — in: {response.usage.prompt_tokens}, out: {response.usage.completion_tokens}")
print(f"Model used: {response.model}")

Response: Hello! How can I help you today? 😊
Tokens — in: 15, out: 11
Model used: google/gemma-4-31b-it-20260402:free


# 🔄 Understanding the ReAct Loop

The **ReAct (Reason + Act)** framework is one of the most widely used approaches for building AI agents. Instead of generating an answer in a single step, the agent repeatedly **reasons**, **takes an action**, **observes the result**, and then decides what to do next.

This iterative process allows the agent to solve tasks that require external information or multiple steps, such as searching Wikipedia, checking the weather, querying databases, or interacting with APIs.

A typical ReAct loop follows this sequence:

```text
User Query
     │
     ▼
 Reason
(What should I do?)
     │
     ▼
 Action
(Call a Tool)
     │
     ▼
 Observation
(Get Tool Result)
     │
     ▼
 Reason Again
(Is more information needed?)
     │
     ▼
 Final Answer
```

The loop continues until the model determines that it has enough information to answer the user's question. If another tool is needed, the cycle repeats. Otherwise, the agent exits the loop and returns the final response.

This reasoning-action cycle makes ReAct agents far more capable than traditional chatbots, as they can gather information, verify results, and make decisions before responding.

### The agent's system prompt

This prompt **IS** the architecture. It tells the model to alternate between THOUGHT and ACTION in a parseable format.

In [19]:
REACT_SYSTEM_PROMPT = """You are a helpful assistant that solves problems step by step using tools.

You have access to these tools:
{tool_descriptions}

## How to respond

When you need to use a tool, respond in EXACTLY this format:

THOUGHT: <your reasoning about what to do next>
ACTION: <tool_name>
ACTION_INPUT: <arguments as valid JSON>

When you have enough information for the final answer:

THOUGHT: <your final reasoning>
FINAL_ANSWER: <your complete answer to the user>

## Rules
- Always start with THOUGHT
- Use only ONE action per turn
- Wait for the OBSERVATION before continuing
- If a tool returns an error, reason about it and try a different approach
- Be concise in your thoughts
"""

### Real tools

Wikipedia search hits a real API. The calculator does real math. These are the "hands" of our agent.

In [20]:
from urllib.parse import quote
from datetime import datetime

# Search Wikipedia
def search_wikipedia(query):
    """Search Wikipedia and return a summary."""

    url = f"https://en.wikipedia.org/api/rest_v1/page/summary/{quote(query)}"

    try:
        r = requests.get(url, timeout=10)

        if r.status_code == 200:
            data = r.json()

            return json.dumps({
                "title": data.get("title", ""),
                "summary": data.get("extract", "No summary found.")[:800]
            })

        return json.dumps({
            "error": f"No Wikipedia page found for '{query}'."
        })

    except Exception as e:
        return json.dumps({
            "error": str(e)
        })


# Calculator
def calculate_math(expression):
    """Safely evaluate a mathematical expression."""

    try:
        allowed = set("0123456789+-*/.() eE")

        if not all(c in allowed for c in expression):
            return json.dumps({
                "error": f"Invalid characters in '{expression}'."
            })

        result = eval(expression)

        return json.dumps({
            "expression": expression,
            "result": round(result, 6)
        })

    except Exception as e:
        return json.dumps({
            "error": str(e)
        })


# Current Date & Time
def get_current_date():
    """Return the current date and time."""

    return json.dumps({
        "datetime": datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    })


# Tool Registry
TOOLS = {
    "search_wikipedia": {
        "fn": search_wikipedia,
        "desc": "Search Wikipedia and return a short summary."
    },

    "calculate": {
        "fn": calculate_math,
        "desc": "Evaluate a mathematical expression."
    },

    "get_current_date": {
        "fn": get_current_date,
        "desc": "Return the current date and time."
    },
}

print("Registered Tools:", list(TOOLS.keys()))

Registered Tools: ['search_wikipedia', 'calculate', 'get_current_date']


### 🔥 THE AGENT LOOP

This is the heart of the class. **~60 lines** that implement the same pattern as Claude Code and Codex.

Walk through it:
1. Build system prompt with tool descriptions
2. Enter a loop (max_iterations prevents infinite loops)
3. Each iteration: call the LLM, get text with THOUGHT + ACTION or FINAL_ANSWER
4. If FINAL_ANSWER → done
5. If ACTION → parse tool name + args, execute the tool
6. Inject OBSERVATION back as a user message
7. Loop

In [21]:
def run_agent(user_query, max_iterations=10, verbose=True):
    """
    A ReAct Agent built from scratch.
    Uses an LLM, external tools, and an agent loop.
    """

    # Build the system prompt with available tool descriptions
    tool_desc = "\n".join(f"- {tool['desc']}" for tool in TOOLS.values())
    system = REACT_SYSTEM_PROMPT.format(tool_descriptions=tool_desc)

    messages = [
        {"role": "system", "content": system},
        {"role": "user", "content": user_query},
    ]

    if verbose:
        print("\n" + "=" * 70)
        print(f"🧑 USER: {user_query}")
        print("=" * 70)

    for i in range(max_iterations):

        if verbose:
            print(f"\n🔄 Iteration {i + 1}/{max_iterations}")

        # STEP 1: Ask the LLM what to do next
        try:
            response = client.chat.completions.create(
                model=FREE_MODEL,
                messages=messages,
                temperature=0,
                max_tokens=800,
            )

        except Exception as e:
            print(f"\n❌ API Error: {e}")
            return {
                "answer": "Failed to communicate with the language model.",
                "iterations": i + 1,
            }

        text = (response.choices[0].message.content or "").strip()

        messages.append({
            "role": "assistant",
            "content": text
        })

        if verbose:
            print("\n🤖 MODEL RESPONSE")
            print("-" * 70)
            print(text)
            print("-" * 70)

        # STEP 2: Extract the THOUGHT
        thought_match = re.search(
            r"THOUGHT:\s*(.+?)(?=ACTION:|FINAL_ANSWER:|$)",
            text,
            re.DOTALL,
        )

        thought = thought_match.group(1).strip() if thought_match else ""

        if verbose and thought:
            print(f"\n💭 THOUGHT: {thought}")

        # STEP 3: Check if the model has finished
        if "FINAL_ANSWER:" in text:

            answer = text.split("FINAL_ANSWER:")[-1].strip()

            if verbose:
                print("\n" + "=" * 70)
                print(f"✅ AGENT FINISHED IN {i + 1} ITERATION(S)")
                print("=" * 70)
                print(answer)

            return {
                "answer": answer,
                "iterations": i + 1,
            }

        # STEP 4: Extract ACTION and ACTION_INPUT
        action_match = re.search(r"ACTION:\s*(\w+)", text)
        input_match = re.search(
            r"ACTION_INPUT:\s*(.+?)(?:\n|$)",
            text,
            re.DOTALL,
        )

        if not action_match:

            if verbose:
                print("\n⚠️ Invalid format returned by the model. Asking again...")

            messages.append({
                "role": "user",
                "content": (
                    "Please respond using either:\n"
                    "ACTION + ACTION_INPUT\n"
                    "or\n"
                    "FINAL_ANSWER."
                ),
            })

            continue

        tool_name = action_match.group(1).strip()
        raw_input = input_match.group(1).strip() if input_match else "{}"

        # STEP 5: Execute the selected tool
        if tool_name not in TOOLS:

            observation = json.dumps({
                "error": f"Unknown tool '{tool_name}'. Available tools: {list(TOOLS.keys())}"
            })

        else:

            try:

                if raw_input.startswith("{"):
                    args = json.loads(raw_input)
                else:
                    args = {"query": raw_input.strip("\"'")}

                observation = TOOLS[tool_name]["fn"](**args)

            except Exception as e:

                observation = json.dumps({
                    "error": f"Failed to execute '{tool_name}': {str(e)}"
                })

        if verbose:
            print(f"\n🔧 ACTION: {tool_name}")
            print(f"📥 INPUT : {raw_input}")

            obs_preview = (
                observation[:300] + "..."
                if len(observation) > 300
                else observation
            )

            print(f"👁️ OBSERVATION: {obs_preview}")

        # STEP 6: Send the observation back to the model
        messages.append({
            "role": "user",
            "content": f"OBSERVATION: {observation}"
        })

    if verbose:
        print(f"\n⚠️ Maximum iterations ({max_iterations}) reached.")

    return {
        "answer": "Maximum iterations reached.",
        "iterations": max_iterations,
    }

### Test — Simple question

Watch the loop: `THOUGHT → ACTION → OBSERVATION → FINAL_ANSWER`

In [22]:
# Test 1: Wikipedia Search
run_agent("Who is Alan Turing?")


🧑 USER: Who is Alan Turing?

🔄 Iteration 1/10

🤖 MODEL RESPONSE
----------------------------------------------------------------------
THOUGHT: The user is asking for information about Alan Turing. I should search Wikipedia to get a comprehensive summary of his life and contributions.
ACTION: search_wikipedia
ACTION_INPUT: {"query": "Alan Turing"}
----------------------------------------------------------------------

💭 THOUGHT: The user is asking for information about Alan Turing. I should search Wikipedia to get a comprehensive summary of his life and contributions.

🔧 ACTION: search_wikipedia
📥 INPUT : {"query": "Alan Turing"}
👁️ OBSERVATION: {"error": "No Wikipedia page found for 'Alan Turing'."}

🔄 Iteration 2/10

🤖 MODEL RESPONSE
----------------------------------------------------------------------

----------------------------------------------------------------------

⚠️ Invalid format returned by the model. Asking again...

🔄 Iteration 3/10

🤖 MODEL RESPONSE
----------------

{'answer': 'Alan Turing was a British mathematician, logician, and computer scientist. He is best known for his work on breaking the Enigma code during World War II, which significantly aided the Allied forces. Turing also made foundational contributions to computer science, including the concept of the Turing machine, which laid the groundwork for modern computing. Additionally, he proposed the Turing test for artificial intelligence. His work has had a profound impact on both theoretical and applied fields.',
 'iterations': 3}

In [23]:
# Test 2: Mathematical Calculation
run_agent("What is (25 + 15) * 4 / 2?")


🧑 USER: What is (25 + 15) * 4 / 2?

🔄 Iteration 1/10

🤖 MODEL RESPONSE
----------------------------------------------------------------------
THOUGHT: Compute (25+15)*4/2.
----------------------------------------------------------------------

💭 THOUGHT: Compute (25+15)*4/2.

⚠️ Invalid format returned by the model. Asking again...

🔄 Iteration 2/10

🤖 MODEL RESPONSE
----------------------------------------------------------------------
THOUGHT: Use evaluate tool to compute expression.ACTION: evaluate
ACTION_INPUT: {"expression":"(25 + 15) * 4 / 2"}
----------------------------------------------------------------------

💭 THOUGHT: Use evaluate tool to compute expression.

🔧 ACTION: evaluate
📥 INPUT : {"expression":"(25 + 15) * 4 / 2"}
👁️ OBSERVATION: {"error": "Unknown tool 'evaluate'. Available tools: ['search_wikipedia', 'calculate', 'get_current_date']"}

🔄 Iteration 3/10

🤖 MODEL RESPONSE
----------------------------------------------------------------------
THOUGHT: Use calculate

{'answer': '80.', 'iterations': 4}

In [24]:
# Test 3: Current Date and Time
run_agent("What is today's date and current time?")


🧑 USER: What is today's date and current time?

🔄 Iteration 1/10

🤖 MODEL RESPONSE
----------------------------------------------------------------------
THOUGHT: The user is asking for the current date and time. I should use the `get_current_time` tool to retrieve this information.
ACTION: get_current_time{}
----------------------------------------------------------------------

💭 THOUGHT: The user is asking for the current date and time. I should use the `get_current_time` tool to retrieve this information.

🔧 ACTION: get_current_time
📥 INPUT : {}
👁️ OBSERVATION: {"error": "Unknown tool 'get_current_time'. Available tools: ['search_wikipedia', 'calculate', 'get_current_date']"}

🔄 Iteration 2/10

🤖 MODEL RESPONSE
----------------------------------------------------------------------
THOUGHT:The user requested today's date and current time. The available tool is `get_current_date`, which likely provides both date and time information. I will use this tool to retrieve the required det

{'answer': "Today's date and time is 2026-06-28 14:50:31.", 'iterations': 3}

In [25]:
# Test 4: Multi-Step Reasoning
run_agent("How many days are left until New Year from today?")


🧑 USER: How many days are left until New Year from today?

🔄 Iteration 1/10

🤖 MODEL RESPONSE
----------------------------------------------------------------------

----------------------------------------------------------------------

⚠️ Invalid format returned by the model. Asking again...

🔄 Iteration 2/10

🤖 MODEL RESPONSE
----------------------------------------------------------------------
THOUGHT: Need current date to calculate days until New Year (Jan 1 of next year). Use the date tool.
ACTION: get_current_date
ACTION_INPUT: {}
----------------------------------------------------------------------

💭 THOUGHT: Need current date to calculate days until New Year (Jan 1 of next year). Use the date tool.

🔧 ACTION: get_current_date
📥 INPUT : {}
👁️ OBSERVATION: {"datetime": "2026-06-28 14:50:57"}

🔄 Iteration 3/10

🤖 MODEL RESPONSE
----------------------------------------------------------------------

----------------------------------------------------------------------

⚠️ Inv

{'answer': 'Maximum iterations reached.', 'iterations': 10}

In [26]:
# Test 6: General Knowledge
run_agent("Tell me about Machine Learning.")


🧑 USER: Tell me about Machine Learning.

🔄 Iteration 1/10

🤖 MODEL RESPONSE
----------------------------------------------------------------------
THOUGHT: The user asks for an overview of Machine Learning. I’ll provide a concise yet comprehensive explanation covering definition, types, key concepts, common algorithms, applications, and current trends.

FINAL_ANSWER: **Machine Learning (ML)** is a subfield of artificial intelligence that focuses on developing algorithms and statistical models that enable computers to learn patterns from data and make decisions or predictions without being explicitly programmed for each specific task.

### Core Ideas
| Concept | Description |
|---------|-------------|
| **Learning from Data** | Algorithms improve their performance on a task as they are exposed to more data. |
| **Model** | A mathematical representation (e.g., a decision tree, neural network) that maps inputs to outputs. |
| **Training** | The process of adjusting a model’s parameters u

{'answer': '**Machine Learning (ML)** is a subfield of artificial intelligence that focuses on developing algorithms and statistical models that enable computers to learn patterns from data and make decisions or predictions without being explicitly programmed for each specific task.\n\n### Core Ideas\n| Concept | Description |\n|---------|-------------|\n| **Learning from Data** | Algorithms improve their performance on a task as they are exposed to more data. |\n| **Model** | A mathematical representation (e.g., a decision tree, neural network) that maps inputs to outputs. |\n| **Training** | The process of adjusting a model’s parameters using a dataset (the *training set*) to minimize error. |\n| **Generalization** | The ability of a trained model to perform well on unseen data (the *test/validation set*). |\n| **Loss Function** | Quantifies the error between the model’s predictions and the true values; the model is optimized to minimize this loss. |\n| **Optimization** | Techniques 

##Trace & Analyze the Agent

Let's instrument the agent to understand **costs, token growth, and behavior patterns**. This is the context engineering problem — every iteration re-sends everything.

In [36]:
def run_agent_traced(user_query, max_iterations=10):
    """Same ReAct agent, but collects detailed execution traces."""

    tool_desc = "\n".join(f"- {t['desc']}" for t in TOOLS.values())
    system = REACT_SYSTEM_PROMPT.format(tool_descriptions=tool_desc)

    messages = [
        {"role": "system", "content": system},
        {"role": "user", "content": user_query},
    ]

    trace = []
    token_log = []

    for i in range(max_iterations):

        try:
            response = client.chat.completions.create(
                model=FREE_MODEL,
                messages=messages,
                temperature=0,
                max_tokens=800,
            )

        except Exception as e:
            trace.append({
                "step": i + 1,
                "type": "❌ API ERROR",
                "error": str(e)
            })
            break

        # Token Usage
        usage = response.usage

        token_log.append({
            "iter": i + 1,
            "in": usage.prompt_tokens,
            "out": usage.completion_tokens,
        })

        text = (response.choices[0].message.content or "").strip()

        messages.append({
            "role": "assistant",
            "content": text
        })

        # Extract Thought
        thought_match = re.search(
            r"THOUGHT:\s*(.+?)(?=ACTION:|FINAL_ANSWER:|$)",
            text,
            re.DOTALL,
        )

        thought = thought_match.group(1).strip() if thought_match else ""

        # Final Answer
        if "FINAL_ANSWER:" in text:

            answer = text.split("FINAL_ANSWER:")[-1].strip()

            trace.append({
                "step": i + 1,
                "type": "✅ FINAL",
                "thought": thought,
                "answer": answer,
            })

            break

        # Extract Action
        action_match = re.search(r"ACTION:\s*(\w+)", text)

        input_match = re.search(
            r"ACTION_INPUT:\s*(.+?)(?:\n|$)",
            text,
            re.DOTALL,
        )

        if not action_match:

            messages.append({
                "role": "user",
                "content": "Please respond using ACTION + ACTION_INPUT or FINAL_ANSWER."
            })

            trace.append({
                "step": i + 1,
                "type": "⚠️ FORMAT",
                "thought": thought,
            })

            continue

        tool_name = action_match.group(1).strip()
        raw_input = input_match.group(1).strip() if input_match else "{}"

        # Execute Tool
        if tool_name in TOOLS:

            try:

                if raw_input.startswith("{"):
                    args = json.loads(raw_input)
                else:
                    args = {"query": raw_input.strip("\"'")}

                observation = TOOLS[tool_name]["fn"](**args)

            except Exception as e:

                observation = json.dumps({
                    "error": str(e)
                })

        else:

            observation = json.dumps({
                "error": f"Unknown tool: {tool_name}"
            })

        trace.append({
            "step": i + 1,
            "type": "🔧 ACTION",
            "thought": thought[:150],
            "tool": tool_name,
            "input": raw_input[:100],
            "observation": observation[:200],
        })

        messages.append({
            "role": "user",
            "content": f"OBSERVATION: {observation}"
        })

    return trace, token_log

In [39]:
trace, tokens = run_agent_traced(
    "Who lived longer — Albert Einstein or Isaac Newton? By how many years?"
)

print("=" * 70)
print("AGENT TRACE")
print("=" * 70)

for step in trace:

    print(f"\nStep {step.get('step')} {step.get('type')}")

    print(f"  Thought : {step.get('thought', '')[:120]}")

    if step.get("type") == "🔧 ACTION":
        print(f"  Tool    : {step.get('tool')}({step.get('input', '')[:60]})")
        print(f"  Result  : {step.get('observation', '')[:120]}")

    elif step.get("answer"):
        print(f"  Answer  : {step.get('answer')[:200]}")

    elif step.get("error"):
        print(f"  Error   : {step.get('error')}")

print("\n" + "=" * 70)
print("TOKEN USAGE PER ITERATION")
print("=" * 70)

total = 0

for t in tokens:

    subtotal = t["in"] + t["out"]
    total += subtotal

    bar = "█" * (t["in"] // 50)

    print(
        f"Iter {t['iter']:>2}: "
        f"{t['in']:>5} in + "
        f"{t['out']:>4} out = "
        f"{subtotal:>5} │ {bar}"
    )

print(f"\nTotal Tokens : {total:,}")
print("Observation  : Prompt tokens increase after every iteration because the")
print("               complete conversation history is sent back to the model.")

AGENT TRACE

Step 1 ⚠️ FORMAT
  Thought : 

Step 2 ✅ FINAL
  Thought : 
  Answer  : Newton lived longer than Einstein by 8 years.

TOKEN USAGE PER ITERATION
Iter  1:   262 in +  117 out =   379 │ █████
Iter  2:   343 in +   34 out =   377 │ ██████

Total Tokens : 756
Observation  : Prompt tokens increase after every iteration because the
               complete conversation history is sent back to the model.


# 📝 Module Summary

In this module, we built our first **ReAct (Reason + Act) Agent** completely from scratch without using frameworks such as LangChain, LangGraph, CrewAI, or AutoGen. Instead of relying on pre-built abstractions, we implemented every step ourselves to understand how an AI agent actually works behind the scenes.

We started by creating a **system prompt** that instructs the language model to follow the ReAct pattern. The model was required to think about the problem, decide whether a tool was needed, generate an action along with its input, wait for the tool's result, and finally produce the answer once enough information had been collected.

Next, we implemented several external tools, including a **Wikipedia search tool**, a **mathematical calculator**, and a **current date and time tool**. These tools simulate the capabilities of an AI agent in the real world, where language models frequently interact with APIs, databases, search engines, and other external services instead of relying only on their internal knowledge.

The core of this module was the **ReAct Loop**. During each iteration, the agent sends the conversation history to the language model. The model reasons about the problem and either requests a tool or provides the final answer. When a tool is requested, our Python program executes the corresponding function, collects the observation, and feeds it back to the model. This cycle continues until the model determines that it has enough information to answer the user's question.

The complete workflow can be summarized as:

```text
User Query
      │
      ▼
Large Language Model
      │
      ▼
THOUGHT
      │
      ▼
ACTION
      │
      ▼
Python Executes Tool
      │
      ▼
OBSERVATION
      │
      ▼
Large Language Model
      │
      ▼
Repeat Until...
      │
      ▼
FINAL ANSWER
```

After building the agent, we extended it by implementing a **tracing system**. Instead of only returning the final answer, the traced version records every reasoning step, selected tool, tool input, observation, and final response. This allows us to inspect how the agent arrived at its conclusion and makes debugging significantly easier.

We also collected **token usage** during every iteration. Since the complete conversation history is sent back to the model after each tool execution, the number of prompt tokens continuously increases. This phenomenon is known as the **Context Engineering Problem**, where growing context leads to higher latency, increased API costs, and larger memory consumption. Understanding this limitation is essential for building scalable AI agents.

Overall, this module demonstrated that an AI agent is not a magical system but a structured loop that repeatedly reasons, performs actions, observes results, and continues until the task is complete. This architecture forms the foundation of modern AI frameworks and autonomous agent systems.

## 🎯 Key Takeaways

- Built a ReAct Agent completely from scratch.
- Learned how the ReAct reasoning loop works.
- Implemented external tools using Python functions.
- Connected the LLM with tools through structured prompts.
- Executed tools based on the model's decisions.
- Returned observations back to the model.
- Built an iterative reasoning loop.
- Recorded every reasoning step using tracing.
- Measured prompt and completion token usage.
- Understood why prompt tokens grow after every iteration.
- Learned the basics of **Context Engineering**, which becomes increasingly important when building production-scale AI agents.

With this foundation, we are now ready to explore **Agent Tracing, Cost Analysis, and Context Engineering** in greater depth and learn how to optimize AI agents for efficiency and scalability.